In [ ]:
import os
import json
import random
import numpy as np
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets

import timm
from timm.data import resolve_data_config, create_transform

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import matplotlib.pyplot as plt

# =========================================================
# CONFIGURATION
# =========================================================
@dataclass
class CFG:
    train_dir = "./DATASET_CP/Train"
    val_dir   = "./DATASET_CP/Val"

    model_name = "swin_small_patch4_window7_224"
    num_classes = 2

    batch_size = 32
    epochs = 40

    lr = 4e-5
    weight_decay = 0.05

    warmup_epochs = 5
    patience = 10

    save_dir = "./checkpoints"
    model_name_best = "lung_swin_classifier_best.pth"
    model_name_last = "lung_swin_classifier_last.pth"
    history_file = "training_history.json"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = torch.cuda.is_available()

    cancer_class_name = "Cancer"


cfg = CFG()
os.makedirs(cfg.save_dir, exist_ok=True)


# =========================================================
# REPRODUCIBILITY
# =========================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything()


# =========================================================
# MODEL DEFINITION
# =========================================================
class ClassificationHead(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 2)
        )

    def forward(self, x):
        return self.layers(x)


base_model = timm.create_model(cfg.model_name, pretrained=True, num_classes=0)
model = nn.Sequential(base_model, ClassificationHead(base_model.num_features)).to(cfg.device)


# =========================================================
# DATA LOADING
# =========================================================
data_cfg = resolve_data_config({}, model=base_model)

transform_train = create_transform(**data_cfg, is_training=True)
transform_eval  = create_transform(**data_cfg, is_training=False)

train_data = datasets.ImageFolder(cfg.train_dir, transform=transform_train)
val_data   = datasets.ImageFolder(cfg.val_dir, transform=transform_eval)

# Weighted sampler to handle class imbalance
targets = [label for _, label in train_data]
class_counts = np.bincount(targets)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(
    train_data,
    batch_size=cfg.batch_size,
    sampler=sampler,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_data,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=4
)

class_names = train_data.classes
cancer_index = {c.lower(): i for i, c in enumerate(class_names)}[cfg.cancer_class_name.lower()]


# =========================================================
# LOSS AND OPTIMIZER
# =========================================================
weights = torch.ones(cfg.num_classes)
weights[cancer_index] = 1.5

criterion = nn.CrossEntropyLoss(weight=weights.to(cfg.device))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay
)


# =========================================================
# LEARNING RATE SCHEDULER (COSINE)
# =========================================================
def cosine_lr(epoch):
    if epoch < cfg.warmup_epochs:
        return cfg.lr * (epoch + 1) / cfg.warmup_epochs

    progress = (epoch - cfg.warmup_epochs) / (cfg.epochs - cfg.warmup_epochs)
    return cfg.lr * 0.5 * (1 + np.cos(np.pi * progress))


# =========================================================
# TRAIN / VALIDATION STEP
# =========================================================
def run_epoch(loader, train=False):
    model.train() if train else model.eval()

    losses = []
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(cfg.device)
        labels = labels.to(cfg.device)

        if train:
            optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        losses.append(loss.item() * images.size(0))

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    loss = np.sum(losses) / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")

    return loss, acc, f1, precision, recall


# =========================================================
# TRAINING LOOP
# =========================================================
history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [],
    "train_f1": [], "val_f1": [],
    "train_precision": [], "val_precision": [],
    "train_recall": [], "val_recall": []
}

best_f1 = -1
early_stop_counter = 0

for epoch in range(cfg.epochs):

    lr = cosine_lr(epoch)
    for g in optimizer.param_groups:
        g["lr"] = lr

    train_loss, train_acc, train_f1, train_prec, train_rec = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_f1, val_prec, val_rec = run_epoch(val_loader, train=False)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)
    history["train_precision"].append(train_prec)
    history["val_precision"].append(val_prec)
    history["train_recall"].append(train_rec)
    history["val_recall"].append(val_rec)

    print(f"Epoch {epoch+1}/{cfg.epochs} | Val F1: {val_f1:.4f}")

    # Save best model
    if val_f1 > best_f1:
        best_f1 = val_f1
        early_stop_counter = 0

        torch.save(
            {"model": model.state_dict(), "epoch": epoch, "best_f1": best_f1},
            os.path.join(cfg.save_dir, cfg.model_name_best)
        )

        print("Saved best model")

    else:
        early_stop_counter += 1

    # Save last model
    torch.save(
        {"model": model.state_dict(), "epoch": epoch, "best_f1": best_f1},
        os.path.join(cfg.save_dir, cfg.model_name_last)
    )

    if early_stop_counter >= cfg.patience:
        print("Early stopping triggered")
        break


# =========================================================
# SAVE TRAINING HISTORY
# =========================================================
with open(os.path.join(cfg.save_dir, cfg.history_file), "w") as f:
    json.dump(history, f)


# =========================================================
# PLOT TRAINING CURVES
# =========================================================
plt.figure(figsize=(14, 10))

plt.subplot(2, 2, 1)
plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Validation")
plt.title("Loss")
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(history["train_f1"], label="Train")
plt.plot(history["val_f1"], label="Validation")
plt.title("F1 Score")
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(history["train_acc"], label="Train")
plt.plot(history["val_acc"], label="Validation")
plt.title("Accuracy")
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(history["train_precision"], label="Train Precision")
plt.plot(history["val_precision"], label="Validation Precision")
plt.plot(history["train_recall"], linestyle="--", label="Train Recall")
plt.plot(history["val_recall"], linestyle="--", label="Validation Recall")
plt.title("Precision & Recall")
plt.legend()

plt.tight_layout()
plt.show()